# 🔄 Day 04a: NAT & IPv6 — Address Translation & Next-Gen Internet

---

## 🎯 What You'll Master Today
- NAT types (Static, Dynamic, PAT/NAPT)
- NAT traversal challenges
- IPv6 addressing & header simplification
- IPv4 → IPv6 transition strategies

---

### 🎭 The Analogy

**NAT** = a company receptionist. Everyone inside has an internal extension (private IP),  
but all calls to the outside world go through one public phone number.  
**IPv6** = switching from 10-digit phone numbers to 39-digit numbers so everyone gets their own.

```
╔══════════════════════════════════════════════════════════════════════╗
║                         NAT IN ACTION                               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Private Network              NAT Router            Internet         ║
║  ┌─────────────┐         ┌──────────────┐      ┌───────────────┐    ║
║  │ 192.168.1.10│─┐       │              │      │               │    ║
║  │ 192.168.1.11│──┼─────→│ 203.0.113.5  │─────→│  8.8.8.8      │    ║
║  │ 192.168.1.12│─┘       │ (Public IP)  │      │  (Google DNS) │    ║
║  └─────────────┘         │              │      └───────────────┘    ║
║                          │ NAT Table:   │                            ║
║                          │ .10:5000→:40001│                          ║
║                          │ .11:5001→:40002│                          ║
║                          │ .12:5002→:40003│                          ║
║                          └──────────────┘                            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. NAT Simulator — PAT (Port Address Translation)
# ============================================

class NATRouter:
    def __init__(self, public_ip):
        self.public_ip = public_ip
        self.nat_table = {}  # (private_ip, private_port) → public_port
        self.reverse_table = {}  # public_port → (private_ip, private_port)
        self.next_port = 40000
    
    def outbound(self, private_ip, private_port, dest_ip, dest_port):
        key = (private_ip, private_port)
        if key not in self.nat_table:
            self.nat_table[key] = self.next_port
            self.reverse_table[self.next_port] = key
            self.next_port += 1
        
        pub_port = self.nat_table[key]
        print(f"    OUTBOUND: {private_ip}:{private_port} → NAT → {self.public_ip}:{pub_port} → {dest_ip}:{dest_port}")
        return pub_port
    
    def inbound(self, src_ip, src_port, public_port):
        if public_port in self.reverse_table:
            priv_ip, priv_port = self.reverse_table[public_port]
            print(f"    INBOUND:  {src_ip}:{src_port} → NAT:{public_port} → {priv_ip}:{priv_port}")
            return priv_ip, priv_port
        else:
            print(f"    INBOUND:  {src_ip}:{src_port} → NAT:{public_port} → ❌ DROPPED (no mapping)")
            return None
    
    def show_table(self):
        print(f"\n  NAT Table ({self.public_ip}):")
        print(f"  {'Internal':>22} {'External':>22}")
        print(f"  {'─'*22} {'─'*22}")
        for (priv_ip, priv_port), pub_port in self.nat_table.items():
            print(f"  {priv_ip + ':' + str(priv_port):>22} {self.public_ip + ':' + str(pub_port):>22}")

nat = NATRouter('203.0.113.5')

print("  ═══ NAT (PAT) Simulation ═══\n")

# Multiple devices sharing one public IP
nat.outbound('192.168.1.10', 5000, '8.8.8.8', 53)    # DNS query
nat.outbound('192.168.1.11', 5001, '142.250.0.1', 443) # HTTPS
nat.outbound('192.168.1.10', 5002, '1.1.1.1', 53)     # Another DNS

nat.show_table()

# Response comes back
print("\n  Response routing:")
nat.inbound('8.8.8.8', 53, 40000)
nat.inbound('142.250.0.1', 443, 40001)
nat.inbound('1.2.3.4', 80, 50000)  # Unknown — dropped

In [ ]:
# ============================================
# 2. NAT Types Deep Dive
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════╗
  ║                      NAT TYPES                                  ║
  ╠══════════════╤═══════════════════════════════════════════════════╣
  ║ Type         │ Description                                      ║
  ╠══════════════╪═══════════════════════════════════════════════════╣
  ║ Static NAT   │ 1:1 mapping. Each private IP gets a fixed       ║
  ║              │ public IP. Used for servers.                     ║
  ║              │ e.g., 192.168.1.10 ←→ 203.0.113.10            ║
  ╠══════════════╪═══════════════════════════════════════════════════╣
  ║ Dynamic NAT  │ Pool of public IPs assigned on-demand.          ║
  ║              │ First-come, first-served. Pool can exhaust.     ║
  ╠══════════════╪═══════════════════════════════════════════════════╣
  ║ PAT / NAPT   │ Many-to-one. All devices share ONE public IP.  ║
  ║ (Overload)   │ Differentiated by port numbers.                  ║
  ║              │ Most common (home routers use this!)             ║
  ╚══════════════╧═══════════════════════════════════════════════════╝

  NAT Problems & Solutions:
  ┌──────────────────┬───────────────────────────────────────────┐
  │ Problem          │ Solution                                  │
  ├──────────────────┼───────────────────────────────────────────┤
  │ Peer-to-peer     │ STUN, TURN servers                       │
  │ Server behind NAT│ Port forwarding / UPnP                   │
  │ VoIP/Gaming      │ NAT traversal (ICE framework)            │
  │ IPsec VPN        │ NAT-T (NAT Traversal for IPsec)          │
  │ FTP active mode  │ ALG (Application Layer Gateway)          │
  └──────────────────┴───────────────────────────────────────────┘
""")

In [ ]:
# ============================================
# 3. IPv6 Addressing
# ============================================

class IPv6Address:
    def __init__(self, addr_str):
        self.original = addr_str
        self.expanded = self._expand(addr_str)
    
    def _expand(self, addr):
        # Handle :: expansion
        if '::' in addr:
            parts = addr.split('::')
            left = parts[0].split(':') if parts[0] else []
            right = parts[1].split(':') if parts[1] else []
            missing = 8 - len(left) - len(right)
            groups = left + ['0000'] * missing + right
        else:
            groups = addr.split(':')
        
        return ':'.join(g.zfill(4) for g in groups)
    
    def compress(self):
        groups = self.expanded.split(':')
        # Remove leading zeros
        groups = [g.lstrip('0') or '0' for g in groups]
        result = ':'.join(groups)
        # TODO: :: compression for longest run of zeros
        return result
    
    def get_type(self):
        if self.expanded.startswith('fe80'):  return 'Link-Local'
        if self.expanded.startswith('fc') or self.expanded.startswith('fd'):
            return 'Unique Local (like private IPv4)'
        if self.expanded.startswith('ff'):    return 'Multicast'
        if self.expanded == '0000:' * 7 + '0001': return 'Loopback (::1)'
        return 'Global Unicast'
    
    def analyze(self):
        print(f"    Original  : {self.original}")
        print(f"    Expanded  : {self.expanded}")
        print(f"    Compressed: {self.compress()}")
        print(f"    Type      : {self.get_type()}")
        print(f"    Bits      : 128 ({len(self.expanded.replace(':', '')) * 4} bits from hex)")

print("  ═══ IPv6 Address Examples ═══\n")
for addr in ['2001:0db8:85a3::8a2e:0370:7334', 'fe80::1', '::1', 'ff02::1']:
    IPv6Address(addr).analyze()
    print()

In [ ]:
# ============================================
# 4. IPv4 vs IPv6 + Transition Strategies
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════════╗
  ║                    IPv4 vs IPv6 COMPARISON                          ║
  ╠═══════════════╤═══════════════════════╤════════════════════════════╣
  ║ Feature       │ IPv4                  │ IPv6                       ║
  ╠═══════════════╪═══════════════════════╪════════════════════════════╣
  ║ Address size  │ 32 bits               │ 128 bits                   ║
  ║ Address space │ 4.3 billion            │ 3.4 × 10³⁸                ║
  ║ Notation      │ Dotted decimal         │ Hex colon (8 groups)      ║
  ║ Header        │ 20-60 bytes (variable) │ 40 bytes (fixed)          ║
  ║ Checksum      │ Yes (in header)        │ No (delegated to L2/L4)  ║
  ║ NAT needed?   │ Yes (IP exhaustion)    │ No (enough for everyone) ║
  ║ Broadcast     │ Yes                    │ No (multicast instead)    ║
  ║ Auto-config   │ DHCP                   │ SLAAC + DHCPv6           ║
  ║ IPsec         │ Optional               │ Mandatory (built-in)     ║
  ║ Fragmentation │ Routers + sender        │ Sender only (PMTUD)     ║
  ╚═══════════════╧═══════════════════════╧════════════════════════════╝

  Transition Strategies:
  ┌────────────────┬────────────────────────────────────────────────┐
  │ Strategy       │ Description                                    │
  ├────────────────┼────────────────────────────────────────────────┤
  │ Dual Stack     │ Run both IPv4 and IPv6 simultaneously.        │
  │                │ Most common. Gradual migration.                │
  ├────────────────┼────────────────────────────────────────────────┤
  │ Tunneling      │ Wrap IPv6 packets inside IPv4 packets.        │
  │ (6to4, Teredo) │ Used when IPv6 islands connect over IPv4.     │
  ├────────────────┼────────────────────────────────────────────────┤
  │ Translation    │ Convert IPv4 ↔ IPv6 at gateway.              │
  │ (NAT64/DNS64)  │ Used when one side is IPv6-only.             │
  └────────────────┴────────────────────────────────────────────────┘
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Why was NAT invented? | IPv4 address exhaustion. NAT lets many private IPs share few public IPs |
| 2 | PAT vs Static NAT? | PAT: many-to-one (port-based). Static: one-to-one (for servers) |
| 3 | Why is IPv6 needed? | IPv4 has only 4.3B addresses. IoT needs trillions. IPv6 has 3.4×10³⁸ |
| 4 | What is dual stack? | Device runs both IPv4 and IPv6 simultaneously. Most common transition strategy |
| 5 | IPv6 header improvements? | Fixed 40 bytes (no options clutter). No checksum. No fragmentation by routers |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • NAT: translates private IP ↔ public IP               │
## │  • PAT: most common, many devices share one public IP    │
## │  • IPv6: 128 bits, no NAT needed, simpler header         │
## │  • Transition: dual stack, tunneling, translation        │
## │  • IPv6 eliminates broadcast (uses multicast instead)    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **SSL/TLS** — Handshake, certificates, HTTPS